# Gamil RAG project

### Step 1: Import library

In [45]:
import os
import ollama
from openai import OpenAI
import json
from dotenv import load_dotenv
from huggingface_hub import login
import gradio as gr
from gmail.gmail_function import login_gmail, get_emails_list
from gmail.ingest import create_embeddings
from tqdm import tqdm
from IPython.display import display, Markdown
from pydantic import BaseModel, Field, model_validator, conint
from typing import Literal
from chromadb import PersistentClient
import numpy as np
from sklearn.manifold import TSNE
import plotly.graph_objects as go
from sentence_transformers import SentenceTransformer
from chromadb import PersistentClient

### Step 2: setting variables and environment

In [36]:
load_dotenv(override = True)
ollama_api_key = os.getenv("OLLAMA_API_KEY")
google_api_key = os.getenv("GOOGLE_API_KEY")
hf_token = os.getenv("HF_TOKEN")
credential_path = r'.credentials/credentials.json' # create credentials from Gmail API
token_path = r'.credentials/token.json' # Token will be created the first time of login to be used for subsequent login
login(hf_token, add_to_git_credential=True)

# # setup ollama
llm_client = OpenAI(base_url = "http://localhost:11434/v1/", api_key = ollama_api_key)
llm_model = "gemma3:1b"

# setup google gemini
gemini = OpenAI(base_url = "https://generativelanguage.googleapis.com/v1beta/openai/", api_key = google_api_key)
gemini_llm_model = "gemini-2.5-flash-lite"

# database and embeddings setting
DB_NAME = "preprocessed_db"
collection_name = "docs"
embedding_model = "text-embedding-3-large"
# setup lms
# llm_client = OpenAI(base_url = "http://localhost:1234/v1/", api_key = "lms")
# llm_model = "google/gemma-3-4b"

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


### Step 3: Check if gmail credentials exist

In [3]:
if not os.path.exists(credential_path):
    print(f"Error: Credentials file '{credential_path}' does not exist.")
else:
    print("Credential file exists!")

Credential file exists!


### Step 4: Login gmail with credentials or token

In [4]:
service = login_gmail(credential_path, token_path)

### Step 5: Retrieve Gmail for the past 7 days

In [5]:
emails = get_emails_list(service, days=7)

### Step 6: Create a summary for every gmail

In [6]:
class Result(BaseModel):
    page_content: str = Field(description = "email sender, date received and summary")
    metadata: dict

class Email(BaseModel):
    title: str = Field(description = "Email title")
    sender: str = Field(description = "email sender details")
    date_received: str = Field(description = "Date of email received")
    body: str = Field(description = "Email body")
    category: Literal["job", "transaction", "reminder", "project", "other"]
    summary: str = Field(description = "Email summary based on title and body")

    def as_result(self):
        metadata = {"sender": self.sender, "date_received": self.date_received, "category" : self.category}
        return Result(page_content = self.sender + " send an email on " + self.date_received + ".\nEmail summary: " + self.summary, metadata = metadata)

class Emails(BaseModel):
    emails: list[Email]

class Email_llm_output(BaseModel):
    category: Literal["job", "transaction", "reminder", "project", "other"]
    summary: str = Field(description = "Email summary based on title and body")

In [44]:
system_prompt = f"""
You will be given the title and body of an email, some emails might not have any plain text for its body.
Based on the information, respond in json format only that contains category and a summary with the structure below.
The category can only strictly be one of the five categories ["job", "transaction", "reminder", "project", "other"].
No other category is allowed.
For example,
{{
  "category": "transaction"
  "summary": string that contains summary of email title and body
}}
"""

def preprocess_emails(emails):
  
  for email in tqdm(emails):
    title = email["title"]
    body = email["body"]
    user_prompt = f""" 
    Please help to provide category and summary of the email based on title and/or body. 
    Keep the summary to one or several sentences that are clear, concise and straight to the point. 

    Here is the title of the email:
    {title}


    {"Here is the body of the email: " + body if body else "This email body does not contain any text. Please categorize and summarize based on the title only."}
    """
    messages = [{"role":"system", "content": system_prompt},
            {"role": "user", "content": user_prompt}]

    # response = llm_client.chat.completions.create(model = llm_model, messages = messages, max_tokens = 2000)
    attempt = 0
    max_attempt = 5
    while attempt < max_attempt:
      attempt += 1
      try:
        response = ollama.chat(model = llm_model, 
                          messages = messages, 
                          format=Email_llm_output.model_json_schema(),
                          options={
                                    'num_predict': 2000,  # This is Ollama's version of max_tokens
                                })
        raw_content = response["message"]["content"]
        data = Email_llm_output.model_validate(json.loads(raw_content))
        if data:
          break

      except Exception as e:
        last_exception = e
        print(f"Attempt {attempt} failed: {e}")
        if attempt == max_attempt:
          print("Max retries reached. Raising exception.")
          raise(last_exception)

    email["category"] = data.category
    email["summary"] = data.summary

  emails = Emails.model_validate({"emails":emails})
  return emails

In [9]:
emails = preprocess_emails(emails)

100%|██████████| 69/69 [02:03<00:00,  1.80s/it]


### Create database based on the Email list

In [10]:
DB_PATH = "preprocessed_db"
collection_name = "docs"
create_embeddings(emails, DB_PATH = DB_PATH, collection_name = collection_name)
    

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2770.77it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Vectorstore created with 69 documents


### Plot embeddings

In [11]:
chroma = PersistentClient(path=DB_NAME)
collection = chroma.get_or_create_collection(collection_name)
result = collection.get(include=['embeddings', 'documents', 'metadatas'])
vectors = np.array(result['embeddings'])
documents = result['documents']
metadatas = result['metadatas']
categories = [metadata['category'] for metadata in metadatas]
colors = [['blue', 'green', 'yellow', 'orange', 'red'][["job", "transaction", "reminder", "project", "other"].index(category)] for category in categories]

In [12]:
tsne = TSNE(n_components=2, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

# Create the 2D scatter plot
fig = go.Figure(data=[go.Scatter(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(categories, documents)],
    hoverinfo='text'
)])

fig.update_layout(title='2D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x',yaxis_title='y'),
    width=800,
    height=600,
    margin=dict(r=20, b=10, l=10, t=40)
)

fig.show()

### Step 8: Create RAG based on the emails

#### Retrieve email with rank order

In [13]:
class RankOrder(BaseModel):
    order: list[conint(ge=1, le=len(emails.emails))]


In [ ]:
def rerank(question, emails_result):
    system_prompt = """
You are a document re-ranker.
You are provided with a question and a list of relevant emails from a query of a knowledge base.
The emails are provided in the order they were retrieved; this should be approximately ordered by relevance, but you may be able to improve on that.
You must rank order the provided emails by relevance to the question, with the most relevant email first.
Reply only with the list of ranked emails ids, nothing else. Include all the emails ids you are provided with, reranked.
Ensure you use each index exactly once.
"""
    user_prompt = f"The user has asked the following question:\n\n{question}\n\nOrder all the emails by relevance to the question, from most relevant to least relevant. Include all the chunk ids you are provided with, reranked.\n\n"
    user_prompt += "Here are the emails:\n\n"
    for index, email in enumerate(emails_result):
        user_prompt += f"# EMAIL ID: {index + 1}:\n\n{email.as_result().page_content}\n\n"
    user_prompt += "Reply only with the list of ranked email ids, nothing else."
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]

    max_attempt = 5
    attempt = 0
    while attempt < max_attempt:
        try:
            attempt += 1
            response = gemini.chat.completions.parse(model=gemini_llm_model, 
                                    messages=messages, 
                                    response_format=RankOrder)
            reply = response.choices[0].message.content
            order = RankOrder.model_validate(json.loads(reply)).order
            if order:
                break
            
        except Exception as e:
            last_exception = e
            print(f"Attempt {attempt} failed: {e}")
            if attempt == max_attempt:
                print("Max retries reached. Raising exception.")
                raise(last_exception)

    return [emails_result[i - 1] for i in order]

In [41]:
question = "What are the companies that offer artificial intelligence job?"
rerank_emails = rerank(question, emails.emails)

In [ ]:
# print(len(emails.emails))
# print(len(rerank_emails))
# print([email.category for email in emails.emails])
# print([email.category for email in rerank_emails])

In [46]:
RETRIEVAL_K = 10

def fetch_context_unranked(question):
    embedding_model = "sentence-transformers/all-MiniLM-L6-v2"
    embedding_client = SentenceTransformer(embedding_model)

    query = embedding_client.encode(question)

    chroma = PersistentClient(path = DB_PATH)
    collection = chroma.get_or_create_collection(collection_name)
    results = collection.query(query_embeddings=[query], n_results=RETRIEVAL_K)
    chunks = []
    for result in zip(results["documents"][0], results["metadatas"][0]):
        chunks.append(Result(page_content=result[0], metadata=result[1]))
    return chunks

In [47]:
chunks = fetch_context_unranked(question)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3548.57it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [49]:
chunks[0].page_content

'LinkedIn Job Alerts <jobalerts-noreply@linkedin.com> send an email on 2026-03-17 12:47:16.\nEmail summary: Highly skilled Artificial Intelligence Engineer seeking a challenging and rewarding role at AMD.  Interested in contributing to the development of machine learning and data analytics solutions.  Experienced with [mention 2-3 key skills - e.g., deep learning, NLP, model optimization].  Looking for a growth opportunity within a dynamic and innovative team.'

#### Retrieve email based on unranked order

In [ ]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] +\
                [{"role":h["role"], "content": h["content"]} for h in history]+\
                    [{"role": "user", "content": message}]
    response = llm_client.chat.completions.create(model = "gemma3:4b", messages = messages)
    return response.choices[0].message.content

In [16]:
with gr.Blocks() as ui:
    with gr.Row():
        gr.ChatInterface(fn=chat)
        
ui.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
